In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
    ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user's primary goal is to gather information about Lunapolis, a fictional setting on the moon, including its capital, weather, population of cheese miners, and the status of a potential union strike.\n\n## SUMMARY\n\n- The capital of the moon is identified as Lunapolis.\n- The weather in Lunapolis is characterized by clear skies, with temperatures ranging from a high of 120°C to a low of -100°C.\n- Lunapolis is home to 100,000 cheese miners.\n- It is suggested that the cheese miners' union may strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nNo further tasks are identified; the information requested by the user has been provided.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='10aa86a5-2864-4e1b-9ee3-018afcfd3c9c'),
              HumanMessage(content="If you were Lunapolis' new president how would y

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user's primary goal is to gather information about Lunapolis, a fictional setting on the moon, including its capital, weather, population of cheese miners, and the status of a potential union strike.

## SUMMARY

- The capital of the moon is identified as Lunapolis.
- The weather in Lunapolis is characterized by clear skies, with temperatures ranging from a high of 120°C to a low of -100°C.
- Lunapolis is home to 100,000 cheese miners.
- It is suggested that the cheese miners' union may strike due to dissatisfaction with the new president.

## ARTIFACTS

None

## NEXT STEPS

No further tasks are identified; the information requested by the user has been provided.


## Trim/delete messages

In [12]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]
    
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [13]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
    
)

In [14]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
    ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='eb9568ee-61e5-4d24-8366-3e0f8d0241fd'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='9e9b23a1-7ff4-41e2-918f-aa24ff17c1a0', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='da259a1b-b3d4-4844-b9f5-69b2480b320a'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='7cdf3b6b-a78e-47c4-81e3-9f83982755bc', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='0e0f1506-7e74-4f1d-8fb6-2e9eb2761f43'),
              AIMessage(content='I can’t read the device’s temperature directly. If you want to

In [15]:
print(response["messages"][-1].content)

I can’t read the device’s temperature directly. If you want to check it, tell me what device you’re using (PC, laptop, phone, tablet, etc.) and whether you can boot it. Here are quick ways to check, depending on the device:

- PC or laptop (if you can boot)
  - Enter BIOS/UEFI at startup (often F2, Del, or Esc) and look for a Hardware Monitor/System Health page to see CPU/GPU temps.
  - If Windows/macOS/Linux is working, use a monitoring tool:
    - Windows: HWInfo, Core Temp, OpenHardwareMonitor
    - macOS: iStat Menus or Macs Fan Control
    - Linux: lm-sensors (sudo apt install lm-sensors; sudo sensors-detect; then sensors)

- Smartphone (iOS/Android)
  - Temperature isn’t typically shown in a simple readout. Check Battery settings for health and be mindful of heat; third-party apps can show temperature but results vary.

- If the device won’t turn on
  - Temperature won’t be measurable without power. Let it cool, then try power cycling (unplug, hold power button 15–20 seconds, plu